## Exact Match Analysis

In thsi notebook we analyze the exact match performance of our best model judge agains RAG. 

In [1]:
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import gc
import torch
import types
import sys
import os
import json
import joblib
from pathlib import Path


from sklearn.linear_model import LogisticRegression
from tqdm import tqdm
from numpy import float64
from sklearn.metrics import roc_auc_score
from utils.metrics.calculate_metric import calculate_agg_metric
from utils.set_random_seed import set_random_seed
# from interpret.glassbox import ExplainableBoostingClassifier
from utils.model_loaders import load_logistic_models_for_subfolder



set_random_seed(42)
EXPERIMENTS = ["experiment_1", "experiment_4", "experiment_54", "experiment_61", "experiment_73"]

INFO 03-18 10:05:44 [__init__.py:216] Automatically detected platform cuda.


In [2]:
def _get_split_sets(df, subfolder, split):
    split_df = df.filter((pl.col("subfolder") == subfolder) & (pl.col("split") == split))

    X_split = []
    y_split = []

    for exp in EXPERIMENTS:
        exp_df = split_df.filter(pl.col("experiment") == exp)

        X_exp = exp_df.select("input").to_numpy()
        X_exp = np.array([i[0] for i in X_exp])
        y_exp = exp_df.select("evaluation").to_numpy()

        # Data is interleaved: first 500 elements go to position 0 of each array,
        # next 500 elements go to position 1 of each array, etc.
        num_arrays = 500

        X_exp_reshaped = []
        y_exp_reshaped = []
        for i in range(num_arrays):
            # Take every 500th element starting from index i
            X_exp_reshaped.append(X_exp[i::num_arrays])
            y_exp_reshaped.append(y_exp[i::num_arrays])

        X_split.append(X_exp_reshaped)
        y_split.append(y_exp_reshaped)

    return X_split, y_split


def get_train_sets(df, subfolder):
    return _get_split_sets(df, subfolder=subfolder, split="train")


def get_test_sets(df, subfolder):
    return _get_split_sets(df, subfolder=subfolder, split="test")

In [3]:
df = []

for subfolder in ["groundtruth", "voting_alt1"]:
    for exp in EXPERIMENTS:
        for split in ["train", "test"]:
            try:
                load_df = pl.read_ipc(f"../binary_collections/{subfolder}/{exp}/{split}.feather")
                load_df = load_df.with_columns([
                    pl.lit(subfolder).alias("subfolder"),
                    pl.lit(exp).alias("experiment"),
                    pl.lit(split).alias("split"),
                ])
                df.append(load_df)
            except Exception as e:
                print(f"Could not load {subfolder}/{exp}/{split}: {e}")
df = pl.concat(df)
df.head()

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Could not memory_map compressed IPC file, defaulting to 

collection_idx,test_idx,input,evaluation,subfolder,experiment,split
i64,i64,"array[i64, 100]",i32,str,str,str
0,0,"[0, 1, … 0]",0,"""groundtruth""","""experiment_1""","""train"""
0,1,"[0, 1, … 0]",1,"""groundtruth""","""experiment_1""","""train"""
0,2,"[0, 1, … 0]",0,"""groundtruth""","""experiment_1""","""train"""
0,3,"[0, 1, … 0]",0,"""groundtruth""","""experiment_1""","""train"""
0,4,"[0, 1, … 0]",1,"""groundtruth""","""experiment_1""","""train"""


## Evaluate models

In [13]:
wiki = pl.read_ipc(f"../../../data/wiki_dump2018_nq_open/processed/wiki.feather")

LOAD = False
if not LOAD:
    dfs_generations = []
    for exp in EXPERIMENTS:
        questions_path = f"../runs/{exp}/questions.feather"
        for file in os.listdir(f"../runs/{exp}/generations"):
            if "lr_groundtruth" in file:
                print(f"Processing {file} for experiment {exp}...")
                dfs_generations.append(calculate_agg_metric(
                    metrics=["squad_v2_best_exact"],
                    generation_path=f"../runs/{exp}/generations/{file}",
                    reference_path=questions_path,
                    saving_path=None            
                )
                .with_columns([
                    pl.lit(exp).alias("experiment"),
                    pl.lit("lr_groundtruth").alias("model"),
                ]))
            
            if "rag" in file:
                print(f"Processing {file} for experiment {exp}...")
                dfs_generations.append(calculate_agg_metric(
                    metrics=["squad_v2_best_exact"],
                    generation_path=f"../runs/{exp}/generations/{file}",
                    reference_path=questions_path,
                    saving_path=None            
                )
                .with_columns([
                    pl.lit(exp).alias("experiment"),
                    pl.lit("rag").alias("model"),
                ]))
            
            if "ebm_voting" in file:
                print(f"Processing {file} for experiment {exp}...")
                dfs_generations.append(calculate_agg_metric(
                    metrics=["squad_v2_best_exact"],
                    generation_path=f"../runs/{exp}/generations/{file}",
                    reference_path=questions_path,
                    saving_path=None            
                )
                .with_columns([
                    pl.lit(exp).alias("experiment"),
                    pl.lit("ebm_voting").alias("model"),
                ]))

    em_results = pl.concat(dfs_generations)
    em_results.write_ipc("em_results.feather")
else:
    em_results = pl.read_ipc("em_results.feather")

Processing rag.json for experiment experiment_1...


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Using the latest cached version of the module from /home/caio.rhoden/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--squad_v2/bd2753381689b3f5bd1f4d85d23b9e2764cf7a26ca1821bcc729f1ee660d1560 (last modified on Mon May 19 17:22:26 2025) since it couldn't be found locally at evaluate-metric--squad_v2, or remotely on the Hugging Face Hub.


Processing ebm_voting.json for experiment experiment_1...


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Using the latest cached version of the module from /home/caio.rhoden/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--squad_v2/bd2753381689b3f5bd1f4d85d23b9e2764cf7a26ca1821bcc729f1ee660d1560 (last modified on Mon May 19 17:22:26 2025) since it couldn't be found locally at evaluate-metric--squad_v2, or remotely on the Hugging Face Hub.


Processing lr_groundtruth.json for experiment experiment_1...


...

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Using the latest cached version of the module from /home/caio.rhoden/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--squad_v2/bd2753381689b3f5bd1f4d85d23b9e2764cf7a26ca1821bcc729f1ee660d1560 (last modified on Mon May 19 17:22:26 2025) since it couldn't be found locally at evaluate-metric--squad_v2, or remotely on the Hugging Face Hub.


Processing lr_groundtruth.json for experiment experiment_73...


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Using the latest cached version of the module from /home/caio.rhoden/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--squad_v2/bd2753381689b3f5bd1f4d85d23b9e2764cf7a26ca1821bcc729f1ee660d1560 (last modified on Mon May 19 17:22:26 2025) since it couldn't be found locally at evaluate-metric--squad_v2, or remotely on the Hugging Face Hub.


Processing rag.json for experiment experiment_73...


Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.
Using the latest cached version of the module from /home/caio.rhoden/.cache/huggingface/modules/evaluate_modules/metrics/evaluate-metric--squad_v2/bd2753381689b3f5bd1f4d85d23b9e2764cf7a26ca1821bcc729f1ee660d1560 (last modified on Mon May 19 17:22:26 2025) since it couldn't be found locally at evaluate-metric--squad_v2, or remotely on the Hugging Face Hub.


In [15]:
em_results.group_by("model").agg(pl.mean("mean"))

model,mean
str,f64
"""lr_groundtruth""",0.446
"""ebm_voting""",0.3192
"""rag""",0.3272
